In [1]:
from pathlib import Path
import sys
# Add root directory to sys.path for module imports
sys.path.append(str(Path().resolve().parent))
from processing.spherical_projection import generate_2D_projection_images
from tools.config_loader import load_config
from tools.pcd_utils import create_dir_if_not_exists
from tools.preprocess_point_cloud import read_and_clean_pcd, map_angle_to_pixel
from tools.spherical_projection_helper import  select_min_range_and_count
import numpy as np
from tools.plot_tools import display_single_band_img_wt_discrete_values

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
config_path = Path("../input_params/3D_to_2D_config_mangrove_roots.json")
current_config = load_config(config_path)

params = current_config['spherical_projection']
global_config = current_config['global']
input_path_ls = global_config['input_path_ls']
output_dir_ls = global_config['output_dir_ls']
v_fov = global_config['v_fov']
h_fov = global_config['h_fov']
canvas_size = global_config['canvas_size']
angular_res = (global_config['v_ang_res_deg'], global_config['h_ang_res_deg'])


('🔹 Loading color map from '
 '/home/fzhcis/mylab/gdrive/projects_with_Jan/for_Fei/palau_2024/labels.json')
("config: {'global': {'input_folder': 'ALRSET1', 'input_folder_parent': "
 "'palau_2024', 'input_base_dir': "
 "'/home/fzhcis/mylab/gdrive/projects_with_Jan/for_Fei', 'output_base_dir': "
 "'/home/fzhcis/mylab/data/mangrove3d/outputs', 'selected_scans': [0, 3], "
 "'input_suffix': '.txt', 'dataset': 'MANGROVE', 'h_fov': [0, 360], 'v_fov': "
 "[0, 135], 'h_ang_res_deg': 0.25, 'v_ang_res_deg': 0.25, "
 "'delete_intermediate_file': True, 'color_map': {'0': (0.0, 0.0, 0.0), '1': "
 "(0.502, 0.0, 0.502), '2': (0.647, 0.165, 0.165), '3': (0.0, 0.502, 0.0), "
 "'4': (1.0, 0.647, 0.0), '5': (1.0, 1.0, 0.0)}, 'output_dir_ls': "
 "[PosixPath('/home/fzhcis/mylab/data/mangrove3d/outputs/ALRSET1/UMBCBL009_2024-03-28-02-47-26_ALRSET12_060180_000200.800_1830507489'), "
 "PosixPath('/home/fzhcis/mylab/data/mangrove3d/outputs/ALRSET1/UMBCBL009_2024-03-28-02-53-31_ALRSET16_060180_000805.800_210262

In [3]:
input_path_ls

[PosixPath('/home/fzhcis/mylab/gdrive/projects_with_Jan/for_Fei/palau_2024/ALRSET1/UMBCBL009_2024-03-28-02-47-26_ALRSET12_060180_000200.800_1830507489.txt'),
 PosixPath('/home/fzhcis/mylab/gdrive/projects_with_Jan/for_Fei/palau_2024/ALRSET1/UMBCBL009_2024-03-28-02-53-31_ALRSET16_060180_000805.800_2102620070.txt'),
 PosixPath('/home/fzhcis/mylab/gdrive/projects_with_Jan/for_Fei/palau_2024/ALRSET1/UMBCBL009_2024-03-28-02-55-21_ALRSET17_060180_000955.799_441453583.txt')]

In [4]:
dataset_name=global_config['dataset']
out_key_str=current_config['calc_geom_feature']['out_signature_str']
color_map=global_config['color_map']
saveflag=params['save_extra_maps']
visualize=params['visualize']
extra_maps=params['extra_maps']
show_single_band=params['show_single_band']
show_pseudo_rgb=params['show_pseudo_rgb']
show_pca=params['show_pca']

In [5]:
for input_path, output_dir in zip(input_path_ls, output_dir_ls):
    # input_file_stem=input_path.stem

    # Setup paths
    img_out_dir = output_dir / 'img'

    create_dir_if_not_exists(img_out_dir, ask_user=False)

    current_file_params = current_config['calc_geom_feature']
    points_df = read_and_clean_pcd(
                input_path, 
                cut_percent=current_file_params["cut_percent"],
                clean_pc=current_file_params["clean_pc"],
                dataset_name=dataset_name,
                flip_mangrove=current_file_params["flip_mangrove"]
            )

    points_df.head()
    azimuth, elevation = points_df['azimuth'], points_df['elevation']
    x_pix, y_pix = map_angle_to_pixel(azimuth, elevation, canvas_size, angular_res)
    points_df['x_pix'] = x_pix
    points_df['y_pix'] = y_pix

    grouped_with_density = points_df.groupby(['y_pix', 'x_pix'], observed=False).apply(
            select_min_range_and_count, include_groups=False
        )

    pts_per_pixel = grouped_with_density['point_count']

    y_coords = np.arange(canvas_size[0])
    x_coords = np.arange(canvas_size[1])

    # Initialize the multi-channel array
    density_map = np.zeros(canvas_size, dtype=np.float32)

    # Extract density indices
    density_y_indices = pts_per_pixel.index.get_level_values(0)
    density_x_indices = pts_per_pixel.index.get_level_values(1)

    density_map[density_y_indices, density_x_indices] = pts_per_pixel.values

    display_single_band_img_wt_discrete_values(
            density_map, 
            title='pt_density_map', 
            output_dir=img_out_dir, 
            saveflag=saveflag,
            visualize=visualize
        )

Reading a text file - for Mongrove roots.
No header detected. Now trying to add predefined column names to the dataframe.
------Flipping of Z axis for mangrove dataset.-------------
Read 815421 points from /home/fzhcis/mylab/gdrive/projects_with_Jan/for_Fei/palau_2024/ALRSET1/UMBCBL009_2024-03-28-02-47-26_ALRSET12_060180_000200.800_1830507489.txt
---------Before preprocessing:----------
Number of points: 815421
Range of X: -34.755 to 34.995; dtype: float64
Range of Y: -37.659 to 34.945; dtype: float64
Range of Z: -0.599 to 21.273; dtype: float64
Range of zenith: 0.000 to 135.000; dtype: float64
Range of azimuth: 0.000 to 359.940; dtype: float64
Range of rangemeter: 0.000 to 43.893; dtype: float64
Range of Intensity: 0.000 to 3951.000; dtype: int64
Range of Return Number: 1.000 to 2.000; dtype: int64
Range of elevation: -90.000 to 45.000; dtype: float64
❗ Cleaning point cloud data based on 0.01% cut-off.
Filtered Intensity: 41884 points removed based on intensity and range limits:
{'ran